# Stress Score Prediction V4 — Tuned ExtraTrees

V3의 누수 없는 전처리와 검증된 행 단위 파생변수를 유지하고, 동일한 5-Fold 검증으로 ExtraTrees의 특성 샘플링 비율을 최적화한 버전입니다.


## 0. Import & Data Load


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score


In [2]:
DATA_DIR = Path('/Users/bitsaem/Desktop/stress_project/open')

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)


Train shape: (3000, 18)
Test shape: (3000, 17)


## 1. Data Split


In [3]:
TARGET = 'stress_score'
ID_COL = 'ID'

X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET].copy()

X_test = test.drop(columns=[ID_COL])
test_ids = test[ID_COL].copy()


## 2. Row-level Feature Engineering


In [4]:
def add_features(df):
    """다른 행이나 데이터셋 통계를 사용하지 않는 행 단위 파생변수 생성 함수."""
    df = df.copy()

    height_m = df['height'] / 100
    sbp = df['systolic_blood_pressure']
    dbp = df['diastolic_blood_pressure']

    df['bmi'] = df['weight'] / (height_m ** 2)
    df['pulse_pressure'] = sbp - dbp
    df['mean_arterial_pressure'] = (sbp + 2 * dbp) / 3
    df['blood_pressure_ratio'] = sbp / (dbp + 1e-6)
    df['cholesterol_glucose_ratio'] = (
        df['cholesterol'] / (df['glucose'] + 1e-6)
    )
    df['missing_count'] = df.isna().sum(axis=1)

    return df


X_v4 = add_features(X)
X_test_v4 = add_features(X_test)

print('Original shape:', X.shape)
print('V4 shape:', X_v4.shape)


Original shape: (3000, 16)
V4 shape: (3000, 22)


## 3. Leakage-safe Preprocessing


In [5]:
categorical_cols = X_v4.select_dtypes(
    include=['object', 'category']
).columns.tolist()

numerical_cols = X_v4.select_dtypes(
    include=['number']
).columns.tolist()

print('Categorical:', categorical_cols)
print('Numerical:', numerical_cols)


Categorical: ['gender', 'activity', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern', 'edu_level']
Numerical: ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure', 'diastolic_blood_pressure', 'glucose', 'bone_density', 'mean_working', 'bmi', 'pulse_pressure', 'mean_arterial_pressure', 'blood_pressure_ratio', 'cholesterol_glucose_ratio', 'missing_count']


In [6]:
categorical_pipeline = Pipeline([
    (
        'imputer',
        SimpleImputer(
            strategy='constant',
            fill_value='missing'
        )
    ),
    (
        'encoder',
        OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=False
        )
    )
])

numerical_pipeline = Pipeline([
    (
        'imputer',
        SimpleImputer(
            strategy='median',
            add_indicator=True
        )
    )
])

preprocessor_v4 = ColumnTransformer([
    ('categorical', categorical_pipeline, categorical_cols),
    ('numerical', numerical_pipeline, numerical_cols)
])


## 4. Tuned ExtraTrees & Cross-validation


In [7]:
model_v4 = ExtraTreesRegressor(
    n_estimators=600,
    min_samples_leaf=1,
    max_features=0.35,
    random_state=42,
    n_jobs=-1
)

pipeline_v4 = Pipeline([
    ('preprocessor', preprocessor_v4),
    ('model', model_v4)
])


In [8]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

v4_mae = -cross_val_score(
    pipeline_v4,
    X_v4,
    y,
    cv=cv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

V2_CV_MAE = 0.208324

V3_CV_MAE = 0.174747

print('V4 Fold MAE:', np.round(v4_mae, 6))
print('V2 Mean MAE:', V2_CV_MAE)
print('V3 Mean MAE:', V3_CV_MAE)
print('V4 Mean MAE:', round(v4_mae.mean(), 6))
print('V4 vs V3:', round(v4_mae.mean() - V3_CV_MAE, 6))


V4 Fold MAE: [0.158584 0.173251 0.171125 0.176932 0.171724]
V2 Mean MAE: 0.208324
V3 Mean MAE: 0.174747
V4 Mean MAE: 0.170323
V4 vs V3: -0.004424


## 5. Final Training & Prediction


In [9]:
pipeline_v4.fit(X_v4, y)

pred_v4 = pipeline_v4.predict(X_test_v4)
pred_v4 = np.clip(pred_v4, 0, 1)

print('Prediction shape:', pred_v4.shape)
print('Prediction range:', pred_v4.min(), pred_v4.max())


Prediction shape: (3000,)
Prediction range: 0.00795 0.9969166666666667


## 6. Submission


In [10]:
submission_v4 = submission.copy()
submission_v4[ID_COL] = test_ids
submission_v4[TARGET] = pred_v4

output_path_v4 = DATA_DIR.parent / 'submit_v4_tuned_extratrees.csv'
submission_v4.to_csv(output_path_v4, index=False)

print('Saved:', output_path_v4)
submission_v4.head()


Saved: /Users/bitsaem/Desktop/stress_project/submit_v4_tuned_extratrees.csv


,ID,stress_score
0,TEST_0000,0.589167
1,TEST_0001,0.920500
2,TEST_0002,0.286317
3,TEST_0003,0.448017
4,TEST_0004,0.520650
